ICD-10 coding pipeline — Step 1: Clinical findings extraction via LLM
*Co-authored with CoCo*

# Step 1: Clinical Findings Extraction

## How it works

1. For each encounter, we feed the **Diagnosis Section** (structured ICD codes, visit diagnoses, active problems, narratives) plus the **HPI** and **PMH** sections to an LLM.
2. The LLM extracts every codeable clinical condition as a structured JSON object with fields: `finding`, `category`, `acuity`, `laterality`, `body_site`, `severity`, `causal_link`, `supporting_quote`, `is_present_on_admission`.
3. The `category` field distinguishes active diagnoses from history-of, symptoms, and rule-outs (critical for correct ICD-10 assignment downstream).
4. The `causal_link` field captures etiology relationships (e.g., "diabetic nephropathy" → causal_link: diabetes) which drive dual-code assignment in Step 3.

## Key design decisions
- **Section-aware:** We pass pre-extracted sections, not raw JSON, to keep the prompt focused
- **Setting-aware:** Outpatient encounters do NOT code rule-outs as diagnoses
- **Structured output:** Each finding is a full JSON object, not just a string
- **Verbatim quotes:** `supporting_quote` provides audit trail back to source

In [ ]:
%%sql -r ctx
-- Context
USE ROLE ACCOUNTADMIN;
USE DATABASE ICD10_CODING_APP;
USE SCHEMA PROCESSING;
USE WAREHOUSE COMPUTE_WH;
ALTER SESSION SET QUERY_TAG = 'icd10_v2:extraction';

---
## Configuration

Adjust these parameters to experiment with different extraction strategies.

In [ ]:
%%sql -r config
-- ============================================================
-- EXTRACTION CONFIG
-- ============================================================
SET EXTRACTION_MODEL = 'claude-4-sonnet';
SET MAX_DX_SECTION_LEN = 8000;
SET MAX_HPI_LEN = 4000;
SET MAX_PMH_LEN = 2000;

In [ ]:
%%sql -r prompt
-- Extraction prompt (stored as session variable for easy iteration)
SET EXTRACTION_PROMPT = '
You are a certified ICD-10-CM coder performing chart abstraction. Extract every codeable clinical condition from this encounter note.

For EACH condition, return a JSON object with these EXACT keys:
{
  "finding": "<clinical term as stated by clinician>",
  "category": "diagnosis|symptom|history_of|ruled_out|suspected",
  "acuity": "acute|chronic|acute_on_chronic|unspecified",
  "laterality": "left|right|bilateral|unspecified|not_applicable",
  "body_site": "<specific anatomical site or null>",
  "severity": "mild|moderate|severe|unspecified",
  "causal_link": "<underlying cause if explicitly stated, or null>",
  "supporting_quote": "<verbatim text from note>",
  "is_present_on_admission": true|false|null
}

CODING RULES:
1. "History of" is NOT an active condition. Mark category="history_of".
2. If a condition has a stated cause (e.g. "diabetic nephropathy"), populate causal_link.
3. Extract EVERY laterality and body site. "Right knee OA" needs laterality=right, body_site=knee.
4. If note says "chronic" or duration >6 weeks, mark acuity=chronic.
5. Do NOT extract vitals, labs, meds, procedures UNLESS they ARE a diagnosis.
6. supporting_quote MUST be a verbatim substring from the document.
7. Return ONLY a JSON array. No markdown fences, no explanation.
';

---
## UDF: Extract Clinical Findings

Wraps the LLM extraction call. Takes section text, returns structured JSON array.

In [ ]:
%%sql -r create_udf
CREATE OR REPLACE FUNCTION PROCESSING.EXTRACT_CLINICAL_FINDINGS(
    DIAGNOSIS_SECTION VARCHAR,
    HPI_SECTION VARCHAR,
    PMH_SECTION VARCHAR,
    ENCOUNTER_SETTING VARCHAR
)
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
    SNOWFLAKE.CORTEX.COMPLETE(
        $EXTRACTION_MODEL,
        CONCAT(
            $EXTRACTION_PROMPT, CHR(10), CHR(10),
            '=== ASSESSMENT/DIAGNOSES SECTION (PRIMARY SOURCE) ===', CHR(10),
            LEFT(COALESCE(DIAGNOSIS_SECTION, ''), $MAX_DX_SECTION_LEN), CHR(10), CHR(10),
            '=== HPI/CONTEXT (SUPPORTING) ===', CHR(10),
            LEFT(COALESCE(HPI_SECTION, ''), $MAX_HPI_LEN), CHR(10), CHR(10),
            '=== PAST MEDICAL HISTORY ===', CHR(10),
            LEFT(COALESCE(PMH_SECTION, 'Not available'), $MAX_PMH_LEN)
        )
    )
$$;

---
## Run Extraction

In [ ]:
%%sql -r run_extraction
-- Run extraction using UDF on all encounters
CREATE OR REPLACE TABLE PROCESSING.ENCOUNTER_DIAGNOSES_RAW AS
SELECT
    s.FILE_NAME,
    s.ENCOUNTER_SETTING,
    PROCESSING.EXTRACT_CLINICAL_FINDINGS(
        s.DIAGNOSIS_SECTION,
        s.HPI_SECTION,
        s.PMH_SECTION,
        s.ENCOUNTER_SETTING
    ) AS EXTRACTION_RAW,
    CURRENT_TIMESTAMP() AS EXTRACTED_AT
FROM PROCESSING.DOCUMENT_SECTIONS s;

In [ ]:
%%sql -r flatten
-- Flatten extracted findings into structured table
CREATE OR REPLACE TABLE PROCESSING.ENCOUNTER_FINDINGS AS
WITH parsed AS (
    SELECT
        FILE_NAME,
        ENCOUNTER_SETTING,
        EXTRACTED_AT,
        TRY_PARSE_JSON(
            CASE
                WHEN EXTRACTION_RAW LIKE '%```json%'
                THEN TRIM(REGEXP_SUBSTR(EXTRACTION_RAW, '```json\\s*(.+?)\\s*```', 1, 1, 's', 1))
                WHEN EXTRACTION_RAW LIKE '%```%'
                THEN TRIM(REGEXP_SUBSTR(EXTRACTION_RAW, '```\\s*(.+?)\\s*```', 1, 1, 's', 1))
                ELSE TRIM(EXTRACTION_RAW)
            END
        ) AS FINDINGS_JSON
    FROM PROCESSING.ENCOUNTER_DIAGNOSES_RAW
)
SELECT
    p.FILE_NAME,
    p.ENCOUNTER_SETTING,
    ROW_NUMBER() OVER (PARTITION BY p.FILE_NAME ORDER BY f.INDEX) AS FINDING_SEQ,
    f.VALUE:finding::VARCHAR AS FINDING,
    f.VALUE:category::VARCHAR AS CATEGORY,
    f.VALUE:acuity::VARCHAR AS ACUITY,
    f.VALUE:laterality::VARCHAR AS LATERALITY,
    f.VALUE:body_site::VARCHAR AS BODY_SITE,
    f.VALUE:severity::VARCHAR AS SEVERITY,
    f.VALUE:causal_link::VARCHAR AS CAUSAL_LINK,
    f.VALUE:supporting_quote::VARCHAR AS SUPPORTING_QUOTE,
    f.VALUE:is_present_on_admission::BOOLEAN AS IS_POA,
    p.EXTRACTED_AT
FROM parsed p,
    LATERAL FLATTEN(input => p.FINDINGS_JSON) f
WHERE p.FINDINGS_JSON IS NOT NULL
    AND f.VALUE:finding IS NOT NULL;

---
## Output Preview

In [ ]:
%%sql -r stats
-- Summary stats
SELECT
    COUNT(*) AS TOTAL_FINDINGS,
    COUNT(DISTINCT FILE_NAME) AS DOCUMENTS_PROCESSED,
    ROUND(COUNT(*)::FLOAT / NULLIF(COUNT(DISTINCT FILE_NAME), 0), 1) AS AVG_FINDINGS_PER_DOC,
    COUNT_IF(CATEGORY = 'diagnosis') AS ACTIVE_DIAGNOSES,
    COUNT_IF(CATEGORY = 'history_of') AS HISTORICAL,
    COUNT_IF(CATEGORY = 'symptom') AS SYMPTOMS,
    COUNT_IF(CAUSAL_LINK IS NOT NULL) AS HAS_CAUSAL_LINK
FROM PROCESSING.ENCOUNTER_FINDINGS;

In [ ]:
%%sql -r sample
-- Sample findings
SELECT FILE_NAME, FINDING_SEQ, FINDING, CATEGORY, ACUITY, LATERALITY, CAUSAL_LINK
FROM PROCESSING.ENCOUNTER_FINDINGS
ORDER BY FILE_NAME, FINDING_SEQ
LIMIT 20;

---
## Done

**Output:** `PROCESSING.ENCOUNTER_FINDINGS` — structured findings per encounter

**Next:** Run `02_search.ipynb` to retrieve ICD-10 candidates for each finding.